In [24]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

import json

import re

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import JsonOutputParser

In [25]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

giga_key = os.getenv("GIGA_KEY")

if not giga_key:

    raise ValueError("Ключ GIGA_KEY не найден в .env")

print("Ключ найден:", giga_key[:8], "...")

Ключ найден: MDE5ZGMz ...


In [26]:
llm = GigaChat(

    credentials=giga_key,

    scope="GIGACHAT_API_PERS",

    model="GigaChat",

    verify_ssl_certs=False,

    temperature=0.2,

    max_tokens=1000,

    timeout=60

)

response = llm.invoke("Привет! Ответь одним коротким предложением.")

print(response.content)

Привет! Коротко и по делу.


In [27]:
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.

Текст заявки: {text}

Верни только число — целое число, соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.

Количество человек:
"""
)

chain = basic_prompt | llm | StrOutputParser()

In [28]:


df = pd.read_csv("rental_26.csv", sep=";")

test_texts = {

    i + 1: text

    for i, text in enumerate(df["text"].head(15))

}
for number, text in test_texts.items():

    result = chain.invoke({"text": text})

    print(f"Заявка №{number}")

    print(f"Текст: {text}")

    print(f"Результат: {result}")

    print("---")


Заявка №1
Текст: Снимем жильё с 1.09по 8.09 двухместный,су в номере ,олимпийская деревня или рядом ,предложения в лс.
Результат: 2
---
Заявка №2
Текст: Ищем недорогое жилье недалеко от моря. 3-местный и 2-местный эконом. 3 взрослых и 3 детей (2,9,11 лет). Строго с 20 по 30 июля
Результат: 6
---
Заявка №3
Текст: Здравствуйте,ищем жилье. 2х местный номер,с удобствами. с 17.07 по 27.07.
Результат: 2
---
Заявка №4
Текст: Здравствуйте. Интересует жилье 3 местный номер.с20 .06 по 28.06 не далеко от моря.
Результат: 3
---
Заявка №5
Текст: Добрый День!! Семья 4 человека, 2 взрослых, дети 10 и 3 года, ищем жилье, можно 3-х местный с доп.местом. С 1 июля поближе к морю и не дорого
😉
Результат: 4
---
Заявка №6
Текст: Здравствуйте. Интересует жильё в Лазаревском. 3е взрослых.
Со своим сан узлом и желательно с балконом. Не больше 10мин до моря.
По приемлемым ценам.
С 4 августа дней на 7-10
Результат: 3
---
Заявка №7
Текст: Ищем жильё эконом класса, до 1000 на двоих, с 7 по 15 августа, недалеко от м

In [29]:
df.head()

,amount,text
0,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но..."
1,6,Ищем недорогое жилье недалеко от моря. 3-местн...
2,2,"Здравствуйте,ищем жилье. 2х местный номер,с уд..."
3,3,Здравствуйте. Интересует жилье 3 местный номер...
4,4,"Добрый День!! Семья 4 человека, 2 взрослых, де..."


In [30]:
df.columns

Index(['amount', 'text'], dtype='str')

In [31]:

results = []

for _, row in df.iterrows():

    text = row["text"]

    try:

        result = chain.invoke({"text": text})

        results.append(result.strip())

    except Exception as e:

        results.append(f"ERROR: {e}")

df["result"] = results

df["result_num"] = pd.to_numeric(df["result"], errors="coerce")

correct = (df["amount"] == df["result_num"]).sum()

total = len(df)

errors = total - correct

accuracy = correct / total


print(f"Всего заявок: {total}")

print(f"Верных ответов: {correct}")

print(f"Ошибок: {errors}")

print(f"Точность: {accuracy:.1%}")


display(df)

df.to_csv("rental_26_with_results.csv", index=False, encoding="utf-8-sig")

Всего заявок: 15
Верных ответов: 15
Ошибок: 0
Точность: 100.0%


,amount,text,result,result_num
0,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но...",2,2
1,6,Ищем недорогое жилье недалеко от моря. 3-местн...,6,6
2,2,"Здравствуйте,ищем жилье. 2х местный номер,с уд...",2,2
3,3,Здравствуйте. Интересует жилье 3 местный номер...,3,3
4,4,"Добрый День!! Семья 4 человека, 2 взрослых, де...",4,4
5,3,Здравствуйте. Интересует жильё в Лазаревском. ...,3,3
6,2,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2,2
7,3,Добрый день. Ищем жильё в Лазаревском. С 10.08...,3,3
8,3,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,3,3
9,5,Добрый день интересует жилье семья 5 человек д...,5,5


In [33]:
json_parser = JsonOutputParser()

advanced_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Ты — эксперт по анализу текстовых заявок на аренду жилья.

Твоя задача — извлекать из заявки структурированную информацию.

Нужно определить:
1. count_adults — количество взрослых
2. count_children — количество детей
3. start_date — дату заезда
4. nights — количество ночей проживания
5. price_per_day — желаемую цену за сутки
6. remarks — особые пожелания

Правила:
- Если указано "2 взрослых и 2 детей", то count_adults = 2, count_children = 2.
- Если указано "семья 5 человек, двое взрослых и трое детей", то count_adults = 2, count_children = 3.
- Если указано "с женой, 2 детей", то count_adults = 2, count_children = 2.
- Если указано только "двухместный номер", но нет детей, считай count_adults = 2, count_children = 0.
- Если указано "3е взрослых", то count_adults = 3, count_children = 0.
- Если дата указана как "с 20 по 30 июля", start_date = "20.07".
- Если дата указана как "с 10.08.2018 по 26.08.2018", start_date = "10.08.2018".
- Если указан диапазон проживания, например "с 20 по 30 июля", nights = 10.
- Если указано "дней на 7-10", бери максимальное значение: nights = 10.
- Если цена указана как "до 1000 на двоих", price_per_day = 1000.
- Если цена не указана, price_per_day = null.
- В remarks записывай пожелания: недалеко от моря, санузел, балкон, бассейн, кондиционер, эконом, под ключ, недорого и т.д.
- Если поле невозможно определить, верни null.
- Верни только JSON без пояснений.
"""
    ),
    (
        "human",
        """
Текст заявки:
{text}

Верни ответ строго в JSON-формате:

{{
  "count_adults": 2,
  "count_children": 1,
  "start_date": "15.08",
  "nights": 8,
  "price_per_day": null,
  "remarks": "рядом с морем"
}}
"""
    )
])

advanced_chain = advanced_prompt | llm | json_parser

In [34]:
df = pd.read_csv("rental_26.csv", sep=";")

advanced_results = []

for _, row in df.iterrows():
    text = row["text"]

    try:
        result = advanced_chain.invoke({"text": text})
        advanced_results.append(result)
    except Exception as e:
        advanced_results.append({
            "count_adults": None,
            "count_children": None,
            "start_date": None,
            "nights": None,
            "price_per_day": None,
            "remarks": f"ERROR: {e}"
        })

advanced_results

[{'count_adults': 2,
  'count_children': 0,
  'start_date': '01.09',
  'nights': 8,
  'price_per_day': None,
  'remarks': 'Олимпийская деревня или рядом, санузел в номере'},
 {'count_adults': 3,
  'count_children': 3,
  'start_date': '20.07',
  'nights': 10,
  'price_per_day': None,
  'remarks': 'недалеко от моря, 3-местный и 2-местный эконом'},
 {'count_adults': 2,
  'count_children': 0,
  'start_date': '17.07',
  'nights': 10,
  'price_per_day': None,
  'remarks': 'рядом с морем'},
 {'count_adults': 2,
  'count_children': 1,
  'start_date': '20.06',
  'nights': 8,
  'price_per_day': None,
  'remarks': 'рядом с морем'},
 {'count_adults': 2,
  'count_children': 1,
  'start_date': '01.07',
  'nights': 8,
  'price_per_day': None,
  'remarks': 'близко к морю, не дорого'},
 {'count_adults': 3,
  'count_children': 0,
  'start_date': '04.08',
  'nights': 10,
  'price_per_day': None,
  'remarks': 'санузел, балкон, рядом с морем'},
 {'count_adults': 2,
  'count_children': 1,
  'start_date': '0

In [37]:
result_df = pd.DataFrame(advanced_results)

df_advanced = pd.concat([df, result_df], axis=1)

display(df_advanced)

df_advanced.to_csv("rental_26_advanced_results.csv", index=False, encoding="utf-8-sig")

,amount,text,count_adults,count_children,start_date,nights,price_per_day,remarks
0,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но...",2,0,01.09,8,NaN,"Олимпийская деревня или рядом, санузел в номере"
1,6,Ищем недорогое жилье недалеко от моря. 3-местн...,3,3,20.07,10,NaN,"недалеко от моря, 3-местный и 2-местный эконом"
2,2,"Здравствуйте,ищем жилье. 2х местный номер,с уд...",2,0,17.07,10,NaN,рядом с морем
3,3,Здравствуйте. Интересует жилье 3 местный номер...,2,1,20.06,8,NaN,рядом с морем
4,4,"Добрый День!! Семья 4 человека, 2 взрослых, де...",2,1,01.07,8,NaN,"близко к морю, не дорого"
5,3,Здравствуйте. Интересует жильё в Лазаревском. ...,3,0,04.08,10,NaN,"санузел, балкон, рядом с морем"
6,2,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2,1,07.08,9,1000.0,рядом с морем
7,3,Добрый день. Ищем жильё в Лазаревском. С 10.08...,2,1,10.08.2018,16,NaN,рядом с морем
8,3,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,2,1,15.08,8,NaN,рядом с морем
9,5,Добрый день интересует жилье семья 5 человек д...,2,3,20.09,10,NaN,рядом с морем


In [38]:
manual_labels = [
    {"true_count_adults": 2, "true_count_children": 0, "true_start_date": "01.09", "true_nights": 7, "true_price_per_day": None},
    {"true_count_adults": 3, "true_count_children": 3, "true_start_date": "20.07", "true_nights": 10, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 0, "true_start_date": "17.07", "true_nights": 10, "true_price_per_day": None},
    {"true_count_adults": 3, "true_count_children": 0, "true_start_date": "20.06", "true_nights": 8, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 2, "true_start_date": "01.07", "true_nights": None, "true_price_per_day": None},
    {"true_count_adults": 3, "true_count_children": 0, "true_start_date": "04.08", "true_nights": 10, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 0, "true_start_date": "07.08", "true_nights": 8, "true_price_per_day": 1000},
    {"true_count_adults": 2, "true_count_children": 1, "true_start_date": "10.08.2018", "true_nights": 16, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 1, "true_start_date": "15.08", "true_nights": 8, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 3, "true_start_date": "20.09", "true_nights": 10, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 0, "true_start_date": "24.07", "true_nights": 8, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 2, "true_start_date": "08.08", "true_nights": 13, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 2, "true_start_date": "08.07", "true_nights": 6, "true_price_per_day": None},
    {"true_count_adults": 1, "true_count_children": 0, "true_start_date": "10.07", "true_nights": 10, "true_price_per_day": None},
    {"true_count_adults": 2, "true_count_children": 2, "true_start_date": "29.06.2019", "true_nights": None, "true_price_per_day": None},
]

manual_df = pd.DataFrame(manual_labels)

df_advanced_checked = pd.concat([df_advanced, manual_df], axis=1)

display(df_advanced_checked)

,amount,text,count_adults,count_children,start_date,nights,price_per_day,remarks,true_count_adults,true_count_children,true_start_date,true_nights,true_price_per_day
0,2,"Снимем жильё с 1.09по 8.09 двухместный,су в но...",2,0,01.09,8,NaN,"Олимпийская деревня или рядом, санузел в номере",2,0,01.09,7.0,NaN
1,6,Ищем недорогое жилье недалеко от моря. 3-местн...,3,3,20.07,10,NaN,"недалеко от моря, 3-местный и 2-местный эконом",3,3,20.07,10.0,NaN
2,2,"Здравствуйте,ищем жилье. 2х местный номер,с уд...",2,0,17.07,10,NaN,рядом с морем,2,0,17.07,10.0,NaN
3,3,Здравствуйте. Интересует жилье 3 местный номер...,2,1,20.06,8,NaN,рядом с морем,3,0,20.06,8.0,NaN
4,4,"Добрый День!! Семья 4 человека, 2 взрослых, де...",2,1,01.07,8,NaN,"близко к морю, не дорого",2,2,01.07,NaN,NaN
5,3,Здравствуйте. Интересует жильё в Лазаревском. ...,3,0,04.08,10,NaN,"санузел, балкон, рядом с морем",3,0,04.08,10.0,NaN
6,2,"Ищем жильё эконом класса, до 1000 на двоих, с ...",2,1,07.08,9,1000.0,рядом с морем,2,0,07.08,8.0,1000.0
7,3,Добрый день. Ищем жильё в Лазаревском. С 10.08...,2,1,10.08.2018,16,NaN,рядом с морем,2,1,10.08.2018,16.0,NaN
8,3,Здравствуйте! Ищем жильё с 15 по 23 августа 2...,2,1,15.08,8,NaN,рядом с морем,2,1,15.08,8.0,NaN
9,5,Добрый день интересует жилье семья 5 человек д...,2,3,20.09,10,NaN,рядом с морем,2,3,20.09,10.0,NaN


In [44]:
import pandas as pd

fields = {
    "count_adults": "true_count_adults",
    "count_children": "true_count_children",
    "start_date": "true_start_date",
    "nights": "true_nights",
    "price_per_day": "true_price_per_day"
}

accuracies = {}

for pred_col, true_col in fields.items():
    pred = df_advanced_checked[pred_col]
    true = df_advanced_checked[true_col]

    if pred_col in ["count_adults", "count_children", "nights", "price_per_day"]:
        pred = pd.to_numeric(pred, errors="coerce")
        true = pd.to_numeric(true, errors="coerce")

        correct_mask = (pred == true) | (pred.isna() & true.isna())
    else:
        pred = pred.fillna("").astype(str).str.strip()
        true = true.fillna("").astype(str).str.strip()

        correct_mask = pred == true

    correct = correct_mask.sum()
    total = len(df_advanced_checked)
    accuracy = correct / total

    accuracies[pred_col] = accuracy

    print(f"{pred_col}: верно {correct} из {total}, точность {accuracy:.1%}")

mean_accuracy = sum(accuracies.values()) / len(accuracies)

print(f"Средняя точность по полям: {mean_accuracy:.1%}")

count_adults: верно 13 из 15, точность 86.7%
count_children: верно 11 из 15, точность 73.3%
start_date: верно 14 из 15, точность 93.3%
nights: верно 8 из 15, точность 53.3%
price_per_day: верно 15 из 15, точность 100.0%
Средняя точность по полям: 81.3%


In [41]:
df_advanced_checked.to_csv("rental_26_advanced_checked.csv", index=False, encoding="utf-8-sig")